# Train the 3di-men model on GPU

Trains the same model as `ml/threedimen/threedimen/model/` (point-cloud
autoencoder + text encoder), just on a real GPU here instead of your local
CPU -- for when the dataset has grown past what's practical to iterate on
locally (hundreds of real Hunyuan3D samples, not tens of mock ones).

**Before running:**
1. On your machine, export a slim training bundle (this drops the raw
   `.stl`/`.glb` copies, which dominate dataset size for real Hunyuan3D
   output at 60k-225k vertices/mesh -- training never reads them anyway):
   ```bash
   cd ml/threedimen
   3dimen dataset export-training-bundle --root data/threedimen_v1_hunyuan --out bundle.zip
   ```
2. On kaggle.com: **Datasets -> New Dataset -> Upload** that `bundle.zip`
   (Kaggle auto-extracts zips into the dataset), name it anything, create it.
3. In this notebook's right sidebar: **Add Input** -> attach that dataset.
4. Settings panel: **Accelerator -> GPU T4 x2** (or P100/whatever's
   available), **Internet -> On**.

**Quality expectation, same as always:** hundreds of samples is still far
below what a generative 3D model needs to fully generalize. GPU + more data
than the original ~40-sample proof of concept should visibly improve
results, but this is not Hunyuan3D-scale quality and shouldn't be presented
as such.

## 0. Confirm GPU + find your uploaded bundle

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU visible -- set Accelerator to GPU T4 x2 or P100, then Factory Reset the session."
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

import glob
manifest_candidates = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert manifest_candidates, (
    "No manifest.json found under /kaggle/input -- did you upload the bundle "
    "zip as a Kaggle Dataset and attach it via 'Add Input'?"
)
DATASET_ROOT = manifest_candidates[0].rsplit("/manifest.json", 1)[0]
print(f"using dataset root: {DATASET_ROOT}")

## 1. Install the small set of packages this needs

In [ ]:
# torch/numpy/scipy are already on Kaggle's base image. Only trimesh isn't.
!pip install -q trimesh
print("installed")

## 2. Model code

Pasted directly (matches `ml/threedimen/threedimen/model/{vocab,networks,chamfer}.py`)
so this notebook doesn't depend on the `3di-men` GitHub repo being public/current.

In [ ]:
from __future__ import annotations
import re
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.nn.functional as F

_TOKEN_RE = re.compile(r"[a-z0-9]+")
PAD = "<pad>"
UNK = "<unk>"

def tokenize(text: str) -> list:
    return _TOKEN_RE.findall(text.lower())

@dataclass
class Vocabulary:
    token_to_id: dict = field(default_factory=dict)

    def __len__(self) -> int:
        return len(self.token_to_id)

    @property
    def pad_id(self) -> int:
        return self.token_to_id[PAD]

    @classmethod
    def build(cls, texts, min_freq: int = 1) -> "Vocabulary":
        counts = {}
        for text in texts:
            for token in tokenize(text):
                counts[token] = counts.get(token, 0) + 1
        tokens = [PAD, UNK] + sorted(tok for tok, c in counts.items() if c >= min_freq)
        return cls(token_to_id={tok: i for i, tok in enumerate(tokens)})

    def encode(self, text: str, max_len: int) -> list:
        unk_id = self.token_to_id[UNK]
        ids = [self.token_to_id.get(tok, unk_id) for tok in tokenize(text)][:max_len]
        ids += [self.pad_id] * (max_len - len(ids))
        return ids

    def to_dict(self) -> dict:
        return {"token_to_id": self.token_to_id}


class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.project = nn.Sequential(nn.Linear(embed_dim, latent_dim), nn.ReLU(), nn.Linear(latent_dim, latent_dim))

    def forward(self, token_ids):
        mask = (token_ids != 0).float().unsqueeze(-1)
        embedded = self.embedding(token_ids) * mask
        pooled = embedded.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return self.project(pooled)


class PointCloudEncoder(nn.Module):
    def __init__(self, hidden_dim, latent_dim):
        super().__init__()
        self.point_mlp = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, points):
        per_point = self.point_mlp(points)
        pooled, _ = per_point.max(dim=1)
        return pooled


class PointCloudDecoder(nn.Module):
    def __init__(self, latent_dim, num_points, hidden_dim):
        super().__init__()
        self.num_points = num_points
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, num_points * 3),
        )

    def forward(self, latent):
        return self.mlp(latent).view(-1, self.num_points, 3)


def chamfer_distance(pred, target):
    dist = torch.cdist(pred, target)
    pred_to_target = dist.min(dim=2).values.mean(dim=1)
    target_to_pred = dist.min(dim=1).values.mean(dim=1)
    return (pred_to_target + target_to_pred).mean()

print("model code loaded")

## 3. Load the training bundle

Mirrors `ml/threedimen/threedimen/model/data.py`'s `load_training_pairs`,
standalone (reads the bundle's manifest.json + each sample's
surface.json/.npz directly, without needing the full `threedimen` package).

In [ ]:
import json
import numpy as np
import trimesh
from pathlib import Path

NUM_POINTS = 2048

def load_surface_points(surface_json_path: Path, num_points: int, rng: np.random.Generator):
    npz_path = surface_json_path.with_suffix(".npz")
    arrays = dict(np.load(npz_path))
    mesh = trimesh.Trimesh(vertices=arrays["patch0:vertices"], faces=arrays["patch0:faces"], process=False)
    points, _ = trimesh.sample.sample_surface(mesh, num_points, seed=rng)
    return np.asarray(points)

def load_training_pairs(root: str, num_points: int, seed: int = 0):
    root = Path(root)
    manifest = json.loads((root / "manifest.json").read_text())
    rng = np.random.default_rng(seed)
    pairs = []
    for sample in manifest["samples"]:
        surface_json = root / f"{sample['surface_representation_path']}.json"
        if not surface_json.with_suffix(".npz").exists():
            continue
        points = load_surface_points(surface_json, num_points, rng)
        pairs.append({"prompt": sample["prompt"], "points": points})
    return pairs

pairs = load_training_pairs(DATASET_ROOT, NUM_POINTS)
print(f"loaded {len(pairs)} training pairs")
print("example:", pairs[0]["prompt"])

## 3b. (Optional) Also pull Cap3D samples directly here on Kaggle

Kaggle has its own internet access, so this can pull real-world (caption,
mesh) pairs from [Cap3D](https://huggingface.co/datasets/tiange/Cap3D)
straight into this session -- no local download / bundle re-export / re-
upload round-trip needed for this specific data source. Mirrors
`ml/threedimen/threedimen/external/cap3d.py`'s logic standalone, same as
the model code above.

**License note** (same as the standalone module): Cap3D's captions are
ODC-By, but the underlying Objaverse assets carry mixed, per-object
licenses -- this does a best-effort permissive-license filter, not a legal
guarantee.

Set `USE_CAP3D = False` below to skip this and train on the uploaded bundle
only.

In [ ]:
USE_CAP3D = True
CAP3D_N = 100
CAP3D_SEED = 0

!pip install -q objaverse huggingface_hub

cap3d_pairs = []
if USE_CAP3D:
    import random
    import pandas as pd
    from huggingface_hub import hf_hub_download
    import objaverse

    _PERMISSIVE_MARKERS = ("by", "cc0", "public-domain", "by-sa")

    def _is_permissive(lic: str) -> bool:
        lic = lic.lower()
        return any(m in lic for m in _PERMISSIVE_MARKERS)

    def _load_annotations_robust(uids):
        # objaverse.load_annotations() raises for the WHOLE batch if even
        # one requested UID can't be resolved -- confirmed by hitting this
        # directly on real Cap3D data. Try the fast batched call first;
        # only fall back to one-at-a-time (so a single bad UID doesn't
        # silently disable license info for every other UID) if the batch
        # call actually fails.
        try:
            return objaverse.load_annotations(uids)
        except Exception:
            pass
        result = {}
        for uid in uids:
            try:
                result.update(objaverse.load_annotations([uid]))
            except Exception:
                continue
        return result

    def _recenter_and_normalize(mesh, target_diagonal=1.0):
        # MUST match geometry/ingest.py's normalization exactly -- the
        # uploaded bundle's points went through that (recentered, scaled to
        # unit bounding-box diagonal). Objaverse objects come in wildly
        # inconsistent real-world scales; without this, mixing raw Cap3D
        # points with the bundle's normalized points in the same training
        # batch makes Chamfer loss dominated by whichever objects happen to
        # have huge coordinates -- confirmed by hitting exactly this
        # (recon/align losses in the thousands to billions, vs. 0.04-0.25
        # normally) on a real run before this fix.
        bounds = mesh.bounds
        center = (bounds[0] + bounds[1]) / 2
        diagonal = float(np.linalg.norm(bounds[1] - bounds[0]))
        mesh = mesh.copy()
        mesh.apply_translation(-center)
        if diagonal > 1e-9:
            mesh.apply_scale(target_diagonal / diagonal)
        return mesh

    captions_path = hf_hub_download(
        repo_id="tiange/Cap3D", filename="Cap3D_automated_Objaverse_full.csv", repo_type="dataset"
    )
    captions_df = pd.read_csv(captions_path, header=None, names=["uid", "caption"])

    rng = random.Random(CAP3D_SEED)
    candidate_uids = captions_df["uid"].tolist()
    rng.shuffle(candidate_uids)

    license_by_uid = {}
    annotations = _load_annotations_robust(candidate_uids[: max(CAP3D_N * 5, 50)])
    for uid, ann in annotations.items():
        license_by_uid[uid] = str((ann or {}).get("license") or "")

    caption_by_uid = dict(zip(captions_df["uid"], captions_df["caption"]))
    selected = []
    for uid in candidate_uids:
        if len(selected) >= CAP3D_N:
            break
        lic = license_by_uid.get(uid, "")
        if lic and not _is_permissive(lic):
            continue
        selected.append(uid)

    # download_processes=1: objaverse.load_objects uses multiprocessing
    # internally, which needs the calling code to be guarded by
    # `if __name__ == "__main__":` on some platforms/launch methods --
    # confirmed by hitting exactly that crash locally with processes=4.
    # Single-process is slower but avoids the whole class of issue.
    downloaded = objaverse.load_objects(uids=selected, download_processes=1)

    rng_np = np.random.default_rng(CAP3D_SEED)
    for uid in selected:
        mesh_path = downloaded.get(uid)
        if mesh_path is None:
            continue
        try:
            mesh = trimesh.load(mesh_path, force="mesh")
            mesh = _recenter_and_normalize(mesh)
            points, _ = trimesh.sample.sample_surface(mesh, NUM_POINTS, seed=rng_np)
            cap3d_pairs.append({"prompt": str(caption_by_uid[uid]), "points": np.asarray(points)})
        except Exception as e:
            print(f"skipping {uid}: {e}")

    print(f"pulled {len(cap3d_pairs)} usable Cap3D samples")

pairs = pairs + cap3d_pairs
print(f"total training pairs (bundle + Cap3D): {len(pairs)}")

## 4. Train

In [ ]:
import time
from torch.utils.data import Dataset, DataLoader

EMBED_DIM = 128
HIDDEN_DIM = 256
LATENT_DIM = 128
MAX_LEN = 32
EPOCHS = 300
BATCH_SIZE = 32
LR = 1e-3
ALIGN_WEIGHT = 1.0
SEED = 0

device = torch.device("cuda")
torch.manual_seed(SEED)

vocab = Vocabulary.build([p["prompt"] for p in pairs])
print(f"vocab size: {len(vocab)}")

class PairDataset(Dataset):
    def __init__(self, pairs, vocab, max_len):
        self.pairs = pairs
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        tokens = torch.tensor(self.vocab.encode(pair["prompt"], self.max_len), dtype=torch.long)
        points = torch.tensor(pair["points"], dtype=torch.float32)
        return tokens, points

dataset = PairDataset(pairs, vocab, MAX_LEN)
loader = DataLoader(dataset, batch_size=min(BATCH_SIZE, len(dataset)), shuffle=True, pin_memory=True)

text_encoder = TextEncoder(len(vocab), EMBED_DIM, LATENT_DIM, pad_idx=vocab.pad_id).to(device)
shape_encoder = PointCloudEncoder(HIDDEN_DIM, LATENT_DIM).to(device)
shape_decoder = PointCloudDecoder(LATENT_DIM, NUM_POINTS, HIDDEN_DIM).to(device)

params = list(text_encoder.parameters()) + list(shape_encoder.parameters()) + list(shape_decoder.parameters())
optimizer = torch.optim.Adam(params, lr=LR)

t0 = time.perf_counter()
for epoch in range(EPOCHS):
    epoch_recon = epoch_align = 0.0
    n_batches = 0
    for tokens, points in loader:
        tokens = tokens.to(device, non_blocking=True)
        points = points.to(device, non_blocking=True)
        optimizer.zero_grad()

        shape_latent = shape_encoder(points)
        recon = shape_decoder(shape_latent)
        recon_loss = chamfer_distance(recon, points)

        text_latent = text_encoder(tokens)
        align_loss = F.mse_loss(text_latent, shape_latent.detach())

        loss = recon_loss + ALIGN_WEIGHT * align_loss
        loss.backward()
        optimizer.step()

        epoch_recon += recon_loss.item()
        epoch_align += align_loss.item()
        n_batches += 1

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:4d}  recon={epoch_recon/n_batches:.5f}  align={epoch_align/n_batches:.5f}  "
              f"elapsed={time.perf_counter()-t0:.1f}s")

print(f"training finished in {time.perf_counter()-t0:.1f}s")

## 5. Save the checkpoint (CPU-portable -- loadable on any machine, GPU or not)

In [ ]:
from dataclasses import asdict

config = {
    "num_points": NUM_POINTS, "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM,
    "latent_dim": LATENT_DIM, "max_len": MAX_LEN, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "lr": LR, "align_weight": ALIGN_WEIGHT, "seed": SEED,
}

checkpoint_path = "/kaggle/working/threedimen_v2.pt"
torch.save(
    {
        "text_encoder": text_encoder.state_dict(),
        "shape_encoder": shape_encoder.state_dict(),
        "shape_decoder": shape_decoder.state_dict(),
        "vocab": vocab.to_dict(),
        "config": config,
    },
    checkpoint_path,
)
print(f"saved {checkpoint_path}")

## 6. Sanity-check generation before downloading anything

In [ ]:
def generate_mesh(prompt: str):
    with torch.no_grad():
        tokens = torch.tensor([vocab.encode(prompt, MAX_LEN)], dtype=torch.long, device=device)
        latent = text_encoder(tokens)
        points = shape_decoder(latent)[0].cpu().numpy()
    return trimesh.Trimesh(vertices=points).convex_hull

for test_prompt in ["a compact aluminum bracket with mounting holes", "a heavy-duty steel pump housing"]:
    mesh = generate_mesh(test_prompt)
    print(f"{test_prompt!r} -> faces={len(mesh.faces)} watertight={mesh.is_watertight}")

## 7. Getting the checkpoint back to your machine

Left sidebar file browser -> `/kaggle/working/` -> `...` menu on
`threedimen_v2.pt` -> **Download**. (Or **Save Version -> Save & Run All**,
then grab it from the notebook's **Output** tab if you want it to persist
past this session.)

Then either:
- **Use it locally**: `3dimen model generate "..." --checkpoint threedimen_v2.pt --out out.stl`
- **Serve it live**: upload it as a new Kaggle Dataset (or a new version of
  the existing one) and point `threedimen_api_server.ipynb` at it, same as
  the original CPU-trained checkpoint -- nothing about the serving notebook
  needs to change, a GPU-trained checkpoint loads and runs on CPU inference
  identically (this model is tiny either way).